# 04 — Modelado y Validación Cruzada
**Fase 4 · CRISP-DM** | Proyecto: Predicción de Stroke

UNAM · Facultad de Ingeniería · Minería de Datos · Semestre 2027-1

---

Modelos evaluados:
- **Regresión Logística** — baseline interpretable
- **Random Forest** — ensamble robusto, captura no-linealidades

Validación: `StratifiedKFold` con **k=5** sobre el conjunto de entrenamiento (post-SMOTE).  
Los modelos entrenados se serializan en `data/`.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, roc_auc_score, recall_score

DATA_DIR     = os.path.join(os.getcwd(), "data")
RANDOM_STATE = 42
CV_FOLDS     = 5

# Cargar conjunto de entrenamiento procesado (post-SMOTE)
train_df = pd.read_csv(os.path.join(DATA_DIR, "processed_train.csv"))
X_train  = train_df.drop(columns=["stroke"])
y_train  = train_df["stroke"]
print(f"Train cargado: {X_train.shape[0]} muestras · {X_train.shape[1]} features")

## 4.1 Definición de modelos

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        solver="lbfgs",
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}
print("Modelos definidos:", list(models.keys()))

## 4.2 Validación cruzada estratificada (5-fold)

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "auc_roc": make_scorer(roc_auc_score, needs_proba=True),
    "recall":  make_scorer(recall_score),
}

cv_results = []
for name, model in models.items():
    print(f"Validando: {name} ...")
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring,
                         return_train_score=False, n_jobs=-1)
    cv_results.append({
        "Modelo":         name,
        "AUC-ROC (mean)": round(np.mean(res["test_auc_roc"]), 4),
        "AUC-ROC (std)":  round(np.std(res["test_auc_roc"]),  4),
        "Recall (mean)":  round(np.mean(res["test_recall"]),  4),
        "Recall (std)":   round(np.std(res["test_recall"]),   4),
    })

cv_df = pd.DataFrame(cv_results)
cv_df

## 4.3 Entrenamiento final y serialización

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    fname = name.lower().replace(" ", "_").replace("-", "")
    path  = os.path.join(DATA_DIR, f"model_{fname}.pkl")
    with open(path, "wb") as f:
        pickle.dump(model, f)
    print(f"Modelo guardado: {path}")

---
✅ **Fase de modelado completada.**  
Continúa en `05_evaluation.ipynb`.